# Housing Price Prediction: Exercises 1 and 2

This notebook follows the California housing example and completes the chapter exercises before refactoring preprocessing into a scikit-learn pipeline. The final section adds a custom transformer for `rooms_per_household`, `bedrooms_per_room`, and `population_per_household`.

The notebook searches upward from the current working directory for `data/raw/housing.csv`. Run `python scripts/ingest_data.py` from the project root first if the file is missing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'data' / 'raw' / 'housing.csv').exists()
    ),
    Path.cwd(),
)
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'housing.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing {DATA_PATH}. Run `python scripts/ingest_data.py` first.'
    )

housing = pd.read_csv(DATA_PATH)
housing.head()

## Initial inspection

In [ ]:
housing.info()
housing.describe().T

In [ ]:
housing.hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

## Stratified train and validation split

Median income is grouped into categories so the validation set preserves the income distribution.

In [ ]:
housing['income_cat'] = pd.cut(
    housing['median_income'],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, validation_index = next(split.split(housing, housing['income_cat']))
train_set = housing.iloc[train_index].drop(columns='income_cat').copy()
validation_set = housing.iloc[validation_index].drop(columns='income_cat').copy()
train_set.shape, validation_set.shape

## Exercise 1: correlation and feature analysis

The strongest linear relationship with median house value can be inspected through the correlation matrix.

In [ ]:
numeric_train = train_set.select_dtypes(include=[np.number])
correlations = numeric_train.corr()['median_house_value'].sort_values(ascending=False)
correlations

In [ ]:
train_set.plot(
    kind='scatter',
    x='longitude',
    y='latitude',
    alpha=0.1,
    s=train_set['population'] / 100,
    c='median_house_value',
    cmap='jet',
    colorbar=True,
    figsize=(10, 7),
)
plt.show()

## Exercise 2: engineered features

Add the three ratio features used in the chapter and inspect their correlations.

In [ ]:
def add_engineered_features(data):
    result = data.copy()
    result['rooms_per_household'] = result['total_rooms'] / result['households']
    result['bedrooms_per_room'] = result['total_bedrooms'] / result['total_rooms']
    result['population_per_household'] = result['population'] / result['households']
    return result

engineered_train = add_engineered_features(train_set)
engineered_train.select_dtypes(include=[np.number]).corr()['median_house_value'].sort_values(ascending=False)

## Pipeline refactor and custom transformer

The transformer below makes feature generation part of the fitted preprocessing pipeline. This prevents train and validation preprocessing from drifting apart.

### Transformer contract

`CombinedAttributesAdder` receives only the numeric columns, in the order listed in `numeric_features` below. The index constants in the transformer must stay aligned with that list; the categorical column is handled separately by `ColumnTransformer`.

In [ ]:
class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    """Add the housing ratio attributes to numeric feature arrays."""

    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rooms_ix = 3
        bedrooms_ix = 4
        population_ix = 5
        households_ix = 6
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        return np.c_[X, rooms_per_household, population_per_household]

In [ ]:
features = train_set.drop(columns='median_house_value')
labels = train_set['median_house_value'].copy()
numeric_features = list(features.select_dtypes(include=[np.number]).columns)
categorical_features = ['ocean_proximity']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attributes', CombinedAttributesAdder()),
    ('scaler', StandardScaler()),
])
full_preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

linear_model = Pipeline([
    ('preprocessor', full_preprocessor),
    ('regressor', LinearRegression()),
])
linear_model.fit(features, labels)
predictions = linear_model.predict(features)
{'rmse': mean_squared_error(labels, predictions) ** 0.5, 'mae': mean_absolute_error(labels, predictions)}

## Compare a tree-based model

The same preprocessing can be reused with a random forest regressor.

In [ ]:
forest_model = Pipeline([
    ('preprocessor', full_preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)),
])
forest_model.fit(features, labels)
validation_features = validation_set.drop(columns='median_house_value')
validation_labels = validation_set['median_house_value']
validation_predictions = forest_model.predict(validation_features)
{'rmse': mean_squared_error(validation_labels, validation_predictions) ** 0.5, 'mae': mean_absolute_error(validation_labels, validation_predictions)}

## Conclusions

1. Stratified sampling keeps the validation income distribution representative.
2. The engineered ratio features expose useful relationships that raw counts hide.
3. A custom transformer makes feature generation reproducible inside the sklearn pipeline.
4. `handle_unknown='ignore'` makes categorical preprocessing robust to unseen validation categories.